# 40 · Validate the load and record data-quality results
**Owner: R · Reviewer: M** · Runs after: everything in 10–32

**What:** prove the warehouse is complete and correct, then measure the "after" side of every data-quality rule.

**Why:** "How did you validate that the ETL loaded the expected records?" is a defence question. The answer is this notebook: every row is accounted for from bronze to gold, totals match, keys are unique and no foreign key is orphaned. **If any check fails, the notebook stops**, so a bad load can never reach the analysis.

Results are saved to `audit.validation_results` and `audit.dq_results` with the run id, so the report can cite them.

In [0]:
%run ./00_common

In [0]:
def q(sql):
    return spark.sql(sql).first()[0]

T_ = lambda layer, name: tname(layer, name)
# count DISTINCT rows, so re-running one notebook inside the same run can't double count its rejects
rej = lambda layer, source: f"(SELECT count(DISTINCT source_row) FROM {T_('audit','rejects')} WHERE run_id = '{RUN_ID}' AND layer = '{layer}' AND source = '{source}')"

print("Validating run:", RUN_ID)

checks = [
    ("Reconciliation", "bronze ICA rows = silver events + silver rejects",
     f"SELECT (SELECT count(*) FROM {T_('bronze','ica')}) - (SELECT count(*) FROM {T_('silver','ica_events')}) - {rej('silver','ica')}"),
    ("Reconciliation", "silver events = gold events + scope exclusions",
     f"SELECT (SELECT count(*) FROM {T_('silver','ica_events')}) - (SELECT count(*) FROM {T_('gold','dim_event')} WHERE event_key > 0) - {rej('gold','ica')}"),
    ("Reconciliation", "gold events = fact rows",
     f"SELECT (SELECT count(*) FROM {T_('gold','dim_event')} WHERE event_key > 0) - (SELECT count(*) FROM {T_('gold','fact_event_loss')})"),
    ("Totals", "normalised loss: silver in-scope = fact",
     f"SELECT abs(coalesce((SELECT sum(normalised_loss_2022_aud) FROM {T_('silver','ica_events')} WHERE in_scope),0) - coalesce((SELECT sum(normalised_loss_2022_aud) FROM {T_('gold','fact_event_loss')}),0))"),
    ("Totals", "original loss: silver in-scope = fact",
     f"SELECT abs(coalesce((SELECT sum(original_loss_aud) FROM {T_('silver','ica_events')} WHERE in_scope),0) - coalesce((SELECT sum(original_loss_aud) FROM {T_('gold','fact_event_loss')}),0))"),
    ("Grain", "one fact row per event", f"SELECT count(*) - count(DISTINCT event_key) FROM {T_('gold','fact_event_loss')}"),
    ("Grain", "one snapshot row per state per year",
     f"SELECT count(*) - count(DISTINCT location_key, date_key) FROM {T_('gold','fact_state_year')}"),
    ("Grain", "snapshot has 8 states x 16 years", f"SELECT count(*) - 8 * 16 FROM {T_('gold','fact_state_year')}"),
    ("Completeness", "no NULL keys in the event fact",
     f"SELECT count(*) FROM {T_('gold','fact_event_loss')} WHERE event_key IS NULL OR start_date_key IS NULL OR location_key IS NULL OR hazard_key IS NULL"),
    ("Validity", "no event starts after today",
     f"SELECT count(*) FROM {T_('gold','fact_event_loss')} f JOIN {T_('gold','dim_date')} d ON d.date_key = f.start_date_key WHERE d.full_date > current_date()"),
    ("Audit", "this run logged steps in bronze, silver and gold",
     f"SELECT 3 - count(DISTINCT layer) FROM {T_('audit','run_log')} WHERE run_id = '{RUN_ID}' AND layer IN ('bronze','silver','gold')"),
    ("Audit", "every logged step balanced (in = out + rejected)",
     f"SELECT count(*) FROM {T_('audit','run_log')} WHERE run_id = '{RUN_ID}' AND status <> 'OK'"),
]
for dim, key in [("dim_date", "date_key"), ("dim_location", "location_key"), ("dim_hazard", "hazard_key"), ("dim_event", "event_key")]:
    checks.append(("Keys", f"{dim}.{key} is unique", f"SELECT count(*) - count(DISTINCT {key}) FROM {T_('gold', dim)}"))
for fact, fk, dim, key in [("fact_event_loss", "event_key", "dim_event", "event_key"), ("fact_event_loss", "start_date_key", "dim_date", "date_key"),
                           ("fact_event_loss", "end_date_key", "dim_date", "date_key"), ("fact_event_loss", "location_key", "dim_location", "location_key"),
                           ("fact_event_loss", "hazard_key", "dim_hazard", "hazard_key"), ("fact_state_year", "location_key", "dim_location", "location_key"),
                           ("fact_state_year", "date_key", "dim_date", "date_key")]:
    checks.append(("Referential integrity", f"{fact}.{fk} has no orphans",
                   f"SELECT count(*) FROM {T_('gold', fact)} f LEFT ANTI JOIN {T_('gold', dim)} d ON d.{key} = f.{fk}"))

results = []
for category, name, sql in checks:
    v = float(q(sql) or 0)
    results.append((RUN_ID, category, name, v, "PASS" if v == 0 else "FAIL", datetime.now()))
vr = spark.createDataFrame(results, "run_id string, category string, check_name string, difference double, result string, checked_at timestamp")
vr.write.format(TABLE_FORMAT).mode("append").saveAsTable(tname("audit", "validation_results"))
display(vr.select("category", "check_name", "difference", "result"))

### Data-quality results: the "after" column of the DQ log

In [0]:
S = T_("silver", "ica_events")
metrics = [
    ("DQ1", "ICA dates moved back 100 years", f"SELECT count(*) FROM {S} WHERE date_century_fixed"),
    ("DQ1", "ICA rows whose year still disagrees with the ICA Year column", f"SELECT count(*) FROM {S} WHERE year(start_date) <> ica_year"),
    ("DQ1b", "ICA events ending before they start (flagged, duration NULL)", f"SELECT count(*) FROM {S} WHERE end_before_start"),
    ("DQ2", "in-scope events: state reported", f"SELECT count(*) FROM {S} WHERE in_scope AND state_source = 'reported'"),
    ("DQ2", "in-scope events: state inferred from text", f"SELECT count(*) FROM {S} WHERE in_scope AND state_source = 'inferred'"),
    ("DQ2", "in-scope events: state unknown (key -1)", f"SELECT count(*) FROM {S} WHERE in_scope AND state_source = 'unknown'"),
    ("DQ2", "in-scope events flagged multi-state", f"SELECT count(*) FROM {S} WHERE in_scope AND multi_state_flag"),
    ("DQ3", "ICA loss values that failed to convert", f"SELECT count(*) FROM {T_('bronze','ica')} b JOIN {S} s ON s.source_row = b._row_num WHERE trim(b.original_loss_value) <> '' AND s.original_loss_aud IS NULL"),
    ("DQ4", "distinct event keys vs distinct CAT codes in scope", f"SELECT count(DISTINCT event_id) - count(DISTINCT cat_code) FROM {S} WHERE in_scope"),
    ("DQ5", "disaster-event dates unparsed after two-format rule", f"SELECT count(*) FROM {T_('silver','disaster_events')} WHERE start_date IS NULL AND NOT placeholder_date"),
    ("DQ6", "state-years with imputed population", f"SELECT count(*) FROM {T_('gold','fact_state_year')} WHERE population_imputed"),
    ("DQ6", "state-years with imputed median income", f"SELECT count(*) FROM {T_('gold','fact_state_year')} WHERE income_imputed"),
    ("DQ7", "events excluded: non-natural hazard", f"SELECT count(*) FROM {S} WHERE exclusion_reason = 'non_natural_hazard'"),
    ("DQ7", "events excluded: outside Australia", f"SELECT count(*) FROM {S} WHERE exclusion_reason = 'outside_australia'"),
    ("DQ7", "events excluded: before 2011", f"SELECT count(*) FROM {S} WHERE exclusion_reason = 'before_scope_start'"),
    ("DQ8", "events whose claim types don't sum to the total (flagged)", f"SELECT count(*) FROM {S} WHERE claims_parts_mismatch"),
    ("DQ13", "ABS footer/note rows rejected", f"SELECT count(DISTINCT source, source_row) FROM {T_('audit','rejects')} WHERE run_id = '{RUN_ID}' AND rule_id = 'DQ13'"),
    ("DQ15", "duplicate ICA rows rejected", f"SELECT count(DISTINCT source_row) FROM {T_('audit','rejects')} WHERE run_id = '{RUN_ID}' AND rule_id = 'DQ15'"),
    ("DQ16", "events whose State column omits states in the name (flagged)", f"SELECT count(*) FROM {S} WHERE state_name_conflict"),
    ("DQ18", "placeholder 1753 dates set to unknown", f"SELECT count(*) FROM {T_('silver','disaster_events')} WHERE placeholder_date"),
    ("Match", "ICA events matched to a disaster event", f"SELECT count(*) FROM {T_('silver','ica_de_match')}"),
    ("Match", "in-scope events enriched with human-impact data", f"SELECT count(*) FROM {T_('gold','dim_event')} WHERE de_matched AND match_confidence <> 'low'"),
]
dq = spark.createDataFrame([(RUN_ID, i, m, float(q(sql) or 0), datetime.now()) for i, m, sql in metrics],
                           "run_id string, dq_id string, metric string, value double, measured_at timestamp")
dq.write.format(TABLE_FORMAT).mode("append").saveAsTable(tname("audit", "dq_results"))
display(dq.select("dq_id", "metric", "value"))

In [0]:
failed = [r for r in results if r[4] == "FAIL"]
assert not failed, f"{len(failed)} validation check(s) failed: {[r[2] for r in failed]}"
print(f"All {len(results)} validation checks passed for run {RUN_ID}.")